# **Import des modules nécessaires**

In [ ]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola et de spacy**


In [48]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",401
1,1865 La confession de Claude.,1865,1,2,Le grillon chantait; le souffle harmonieux des...,336
2,1865 La confession de Claude.,1865,1,3,"Pars cependant, puisque tu as soif de la vie. ...",248
3,1865 La confession de Claude.,1865,1,4,"Ai-je eu trop de confiance en ma force, ma pla...",336
4,1865 La confession de Claude.,1865,1,5,Le premier rayon a chassé le cauchemar de ma v...,359


In [67]:
nlp = spacy.load("fr_core_news_lg") #chargement du modèle de langue français de spacy

nlp.max_length = 2_000_000  

In [49]:
df.shape

(10638, 6)

# **Traitement du Corpus de Zola**

In [68]:
def clean_for_topics(text):
    doc = nlp(text)
    mots_propres = []
    
    for token in doc:
        # Exclusion de la ponctuation, des stop words, des nombres et des Entités Nommées
        if token.is_punct or token.is_stop or token.like_num or token.ent_type_ in ['PER', 'LOC', 'ORG']:
            continue
            
        # Conservation et lemmatisation des Noms, Adjectifs et Verbes
        if token.pos_ in ['NOUN', 'ADJ', 'VERB']:
            mots_propres.append(token.lemma_.lower())
            
    return " ".join(mots_propres)

# Application sur ton dataframe existant
print("Nettoyage en cours pour la représentation des topics...")
df['texte_nettoye'] = df['texte'].apply(clean_for_topics)
df.head()

Nettoyage en cours pour la représentation des topics...


,roman,annee,ordre_romans,paquet_id,texte,nb_mots,topic,texte_nettoye
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",401,-1,hiver air matin devenir frais mettre manteau b...
1,1865 La confession de Claude.,1865,1,2,Le grillon chantait; le souffle harmonieux des...,336,-1,grillon chanter souffle harmonieux nuit bercer...
2,1865 La confession de Claude.,1865,1,3,"Pars cependant, puisque tu as soif de la vie. ...",248,-1,soif vie projet soi ferme loyal action rêve vi...
3,1865 La confession de Claude.,1865,1,4,"Ai-je eu trop de confiance en ma force, ma pla...",336,136,confiance force place rêver côté jour naître p...
4,1865 La confession de Claude.,1865,1,5,Le premier rayon a chassé le cauchemar de ma v...,359,136,rayon chasser cauchemar veille obstacle accept...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [64]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.27k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/463 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [65]:
print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(), show_progress_bar=True)

Génération des embeddings sémantiques...


Batches:   0%|          | 0/333 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

In [51]:
docs = df["texte"].tolist()

timestamps = df["ordre_romans"].tolist()

### 1) HDBSCAN

In [85]:
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=1,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

### 2) UMAP

In [83]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

### 2) HDBSCAN

### 3) creation des stop words + vectorisation TF-IDF

In [71]:
stop_word = [
    # noms familiaux centraux
    "rougon", "macquart", "saccard", "mouret", "lantier", "coupeau", "maheu", "fouan", "quenu", "chanteau", "josserand", "deberle",
    "raquin", "lorilleux", "buteau", "roubaud", "baudu", "gervaise",

    # Prénoms / personnages très fréquents
    "pierre", "jean", "jacques", "marie", "madeleine", "pauline", "hélène", "jeanne", "claude", "marius", "lazare", "guillaume",
    "laurent", "philippe", "thérèse", "albine", "maurice", "marthe", "maxime", "daniel", "camille", "serge", "octave", "florent",
    "félicité", "louise", "silvère", "caroline", "lisa", "laurence", "miette", "simon", "josine", "clotilde", "véronique", "juliette",
    "rosalie", "mathieu", "charles", "françoise", "lise", "séverine", "henriette", "georges", "angélique", "adèle", "pascal", "lucie", 
    "martine", "virginie", "suzanne", "valentine", "olympe", "lucien", "armande", "constance", "berthe", "sidonie", "benedetta",

    # prénom / personnages complémentaires
    "luc", "geneviève", "catherine", "chaval", "adélaïde", "antoine", "auguste", "marianne", "clorinde", "normande", "saget", "mlle saget",
    "delhomme", "ragu", "trouche", "sandoz", "mahoudeau", "bonneville", "rastoil", "rambaud", "fenil", "surin", "malignon", "faloise",
    "lambesc", "mignot", "baugé", "morange", "bouchard", "teuse", "pâquerette", "paquerette", "duparque", "mme duparque", "guersaint",
    "aurélie", "aurelie", "mme aurélie", "mme aurelie", "lecœur", "lecoeur", "mme lecœur", "mme lecoeur", "granoux", "bourron",
    "lerat", "mme lerat", "duveyrier", "bachelard", "gueulin", "delaherche", "beauclair", "lison", "orlando", "delbos",
    "boisgelin", "fernande", "sébastien", "sebastien", "mazaud", "dubuche", "moreux", "vigneron", "bourrette", "satan",
    "archangias", "férat", "ferat", "barbue", "trouille", "laveuve",

    # Personnages secondaires / noms très discriminants
    "fauchery", "muffat", "bordenave", "zoé", "mignon", "faujas", "mathéus", "trublot", "fagerolles", "cazalis", "poizat", "jordan",
    "vandeuvres", "condamin", "paloque", "charbonnel", "gilquin", "marjolin", "campardon", "toussaint", "grivet", "denizet", "kahn",
    "hutin", "bonnaire", "laboque", "marsy", "nanet", "favier", "deloche", "goujet", "cadine", "chouteau", "rochas", "lepailleur",
    "séguin", "gagnière", "hourdequin", "macqueron", "théodose", "savin", "gérard", "larsonneau", "morfain", "lorin", "salvat",
    "boche", "gavard", "martelly",

    # Groupes nominaux religieux / institutionnels
    "sœur hyacinthe", "soeur hyacinthe", "mère chantemesse", "mere chantemesse",

     # Formes d’adresse / famille
    "monsieur", "madame", "mademoiselle", "mme", "mlle",
    "maman", "papa", "père", "pere", "mère", "mere",
    "frère", "frere", "frères", "freres", "sœur", "soeur",
    "sœurette", "soeurette", "tante", "oncle", "cousin",
    "cousine", "époux", "epoux", "épouse", "epouse",
    "nièce", "niece",

    # Mots très fréquents et peu informatifs
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "au", "aux",
    "ce", "cet", "cette", "ces", "son", "sa", "ses", "leur", "leurs",
    "je", "tu", "il", "elle", "nous", "vous", "ils", "elles",
    "me", "te", "se", "moi", "toi", "lui", "eux",
    "que", "qui", "quoi", "dont", "où", "et", "ou", "mais", "donc",
    "or", "ni", "car", "ne", "pas", "plus", "moins", "très",
    "dans", "sur", "sous", "avec", "sans", "pour", "par", "comme",
    "chez", "vers", "entre", "contre", "depuis", "pendant",
    "tout", "tous", "toute", "toutes", "même", "autre", "autres",
    "être", "avoir", "faire", "dire", "aller", "voir", "savoir",
    "pouvoir", "vouloir", "falloir", "devoir", "venir", "prendre",
    "mettre", "donner", "trouver", "passer", "rester", "sembler",
    "encore", "toujours", "jamais", "bien", "mal", "peu", "assez",
    "alors", "puis", "enfin", "aussi", "ainsi", "déjà", "là",
    "ici", "cela", "ça", "ceci", "celui", "celle", "ceux", "celles"
]

In [72]:
vectorizer_model = CountVectorizer(
    stop_words=stop_word,
    min_df=5, #
    max_df=0.50 #   
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)

### 4) Création du modèle BERTopic

In [86]:
topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)
topics, probs = topic_model.fit_transform(df["texte_nettoye"].tolist(), embeddings= embeddings)



2026-06-03 11:22:09,381 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-03 11:22:11,555 - BERTopic - Dimensionality - Completed ✓
2026-06-03 11:22:11,556 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-03 11:22:12,070 - BERTopic - Cluster - Completed ✓
2026-06-03 11:22:12,076 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-03 11:22:12,978 - BERTopic - Representation - Completed ✓


## **3) Topics Présent**

In [87]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5147,-1_tableau_député_ministre_gendarme,"[tableau, député, ministre, gendarme, planche,...",[arrêter regret fils traverser angoisse jour t...
1,0,340,0_volupté_affection_solitude_tiède,"[volupté, affection, solitude, tiède, étreinte...",[fond dévorer impatience raidir tendre nerf vi...
2,1,302,1_écoute_étreinte_sanglot_laisse,"[écoute, étreinte, sanglot, laisse, chéri, veu...",[être rester vouloir vouloir savoir expliquer ...
3,2,239,2_science_social_humanité_pape,"[science, social, humanité, pape, universel, c...",[doute torturer rajeunissement antique catholi...
4,3,201,3_oreiller_couverture_bougie_clef,"[oreiller, couverture, bougie, clef, tiroir, s...",[maman reprendre donner chaise long lit bête e...
...,...,...,...,...,...
132,131,15,131_brigadier_majesté_ministre_notaire,"[brigadier, majesté, ministre, notaire, sabre,...",[-vou remettre particulier pied attendre répon...
133,132,15,132_plume_panier_art_cave,"[plume, panier, art, cave, moineau, raisin, pa...",[poule coq coucher côté bel innocence emplir g...
134,133,15,133_battoir_comtesse_baquet_seau,"[battoir, comtesse, baquet, seau, colique, alc...",[déraisonnable ignoble mettre état pareil hont...
135,134,15,134_libéral_hypothéquer_erreur_bongard,"[libéral, hypothéquer, erreur, bongard, cercle...",[falloir répondre bon déclarer air tranquille ...


In [89]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['texte_nettoye'].tolist(), 
    timestamps, 
    nr_bins=20 # Optionnel : à ajuster selon le nombre d'années distinctes de ton corpus
)

# Visualisation interactive de l'évolution des 10 premiers topics
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)

19it [00:03,  6.30it/s]


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'hovertext': [<b>Topic 0</b><br>Words: mission, dévouement,
                            solitude, affection, tâche, <b>Topic 0</b><br>Words:
                            nerf, noiraud, volupté, affection, effroi, <b>Topic
                            0</b><br>Words: calèche, volupté, tant, complice,
                            langueur, <b>Topic 0</b><br>Words: pêche, groseille,
                            cerise, poire, trapu, <b>Topic 0</b><br>Words: lis,
                            cueillir, prairie, chat, canard, <b>Topic
                            0</b><br>Words: tousser, deberl, bouillotte, anxiété,
                            anéantissement, <b>Topic 0</b><br>Words: catin, odieux,
                            landau, restaurant, engueuler, <b>Topic 0</b><br>Words:
                            réconciliation, mardi, répugnance, cuisinière, dépaysé,
                            <b>Topic 0</b><br>Words: affection, bougie, fiévreux,
                            maladie, volet, <b>Topic 0</b><br>Words: chandelle,
                            sorbier, goyot, échelon, fosse, <b>Topic
                            0</b><br>Words: peinture, pont, camaraderie, flèche,
                            avenue, <b>Topic 0</b><br>Words: monseigneur,
                            cathédrale, aiguille, péché, serment, <b>Topic
                            0</b><br>Words: tunnel, meurtre, banque, haie, tiède,
                            <b>Topic 0</b><br>Words: dossier, hérédité, psyché,
                            apen, pendule, <b>Topic 0</b><br>Words: maladie,
                            guérie, bouillon, adorable, seigneur, <b>Topic
                            0</b><br>Words: soif, ambassade, surhumain, messe,
                            vertige, <b>Topic 0</b><br>Words: maternité, opération,
                            solitude, opérer, empoisonner, <b>Topic 0</b><br>Words:
                            couple, halle, allégresse, âgé, roche, <b>Topic
                            0</b><br>Words: servage, ajourner, conformer,
                            sensualité, culte],
              'marker': {'color': '#E69F00'},
              'mode': 'lines',
              'name': '0_volupté_affection_solitude_tiède',
              'type': 'scatter',
              'x': {'bdata': ('AiuHFtkjnUCamZmZmSudQM3MzMzMOp' ... 'MzM6WdQM3MzMzMrJ1AZmZmZma0nUA='),
                    'dtype': 'f8'},
              'y': {'bdata': 'FFAQAg0YBAIaBhMSGBQPBRURCA==', 'dtype': 'i1'}},
             {'hoverinfo': 'text',
              'hovertext': [<b>Topic 1</b><br>Words: mourant, apaisement, infâme,
                            cruel, infamie, <b>Topic 1</b><br>Words: étreinte,
                            cruel, bohémien, passé, lâcheté, <b>Topic
                            1</b><br>Words: tant, coupé, étreinte, moulin, tocsin,
                            <b>Topic 1</b><br>Words: resserre, supposer, poule,
                            héritage, légume, <b>Topic 1</b><br>Words: écoute,
                            parterre, parfum, feuillage, alcôve, <b>Topic
                            1</b><br>Words: baisa, bouché, content, sanglot,
                            ronfler, <b>Topic 1</b><br>Words: diamant, ronflement,
                            partage, accouder, écoute, <b>Topic 1</b><br>Words:
                            rater, paroissien, gaz, marié, gifle, <b>Topic
                            1</b><br>Words: veu, étreinte, angine, toux,
                            fanfaronnade, <b>Topic 1</b><br>Words: injustice,
                            écoute, cage, maheude, brocanteur, <b>Topic
                            1</b><br>Words: tableau, peinture, échelle, maudire,
                            pose, <b>Topic 1</b><br>Words: punition, soumettre,
                            empourprer, vitrail, broderie, <b>Topic 1</b><br>Words:
                            chéri, écoute, embrasse, bague, oubli, <b>Topic
                            1</

In [88]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('volupté', np.float64(0.007313906920985119)), ('affection', np.float64(0.007283256832130206)), ('solitude', np.float64(0.006521384957157691)), ('tiède', np.float64(0.0062819219839529705)), ('étreinte', np.float64(0.005528118763126145)), ('meurtre', np.float64(0.005298741041785427)), ('passé', np.float64(0.005292815339306285)), ('nerf', np.float64(0.005128924277715361)), ('déchirer', np.float64(0.004966210367856219)), ('maladie', np.float64(0.004680271390563365))]

TOPIC 1
[('écoute', np.float64(0.012314660466352295)), ('étreinte', np.float64(0.012289235743916431)), ('sanglot', np.float64(0.009505847921192457)), ('laisse', np.float64(0.008842439833508398)), ('chéri', np.float64(0.00857232506737913)), ('veu', np.float64(0.008226589977665233)), ('tai', np.float64(0.006960521891359497)), ('cruel', np.float64(0.006544827037940918)), ('affection', np.float64(0.006167992371452063)), ('poison', np.float64(0.006109693370025943))]

TOPIC 2
[('science', np.float64(0.01313649963164348))